# Step 5: Work with GeoJSON Regions and Corridors


GeoJSON polygons let you ask targeted spatial questions: which stations are inside a basin, which event-station paths cross a boundary, and how residuals behave inside a corridor. In this notebook, you will load the example regions, make regional residual plots, build corridor selections, and inspect the waveforms behind one boundary-crossing subset.


## Imports

These imports load GeoJSON regions, build corridor selections, and create the maps and record sections used in this notebook.

In [ ]:
from pathlib import Path
import runpy

# Make the local source checkout importable when running notebooks without an installed wheel.
_bootstrap = next(
    (
        path
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for path in (
            candidate / "_source_bootstrap.py",
            candidate / "docs" / "examples" / "_source_bootstrap.py",
        )
        if path.exists()
    ),
    None,
)
if _bootstrap is None:
    raise RuntimeError("Could not find docs/examples/_source_bootstrap.py.")
repo_root = runpy.run_path(str(_bootstrap))["use_source_checkout"]()

from spatial_vtk.config import (
    notebook_timer,
    notebook_figure_settings,
    prepare_notebook_geospatial_environment,
    register_svtk_cell_timer,
)
prepare_notebook_geospatial_environment()

with notebook_timer():
    from pathlib import Path
    import os
    import matplotlib.pyplot as plt
    from IPython.display import Markdown, display

    from spatial_vtk.config.labels import metric_display_name, model_display_name
    from spatial_vtk.io import (
        event_ids_from_records,
        event_label_preview_frame,
        event_rows_for_records,
        first_nonempty_table_value,
        load_configured_input_paths,
        load_configured_input_tables,
        output_group,
    )
    from spatial_vtk.qc import build_qc_waveform_comparison_records
    from spatial_vtk.spatial import (
        BoundaryCorridorConfig,
        CorridorAnchorConfig,
        CorridorSelectionConfig,
        annotate_points_with_geojson,
        build_boundary_corridors,
        build_metric_field,
        corridor_record_pair_frame,
        corridor_record_preview_frame,
        event_station_records_matching_pairs,
        geojson_matched_record_frame,
        classify_paths_with_geojson,
        geojson_metric_region_frame,
        geojson_metric_subset_frame,
        geojson_polygon_preview_table,
        run_geojson_region_summary_workflow_from_config,
        select_records_by_corridors,
        summarize_station_bias,
    )
    from spatial_vtk.spatial.map import plot_corridor_map, plot_geojson_polygons_map, plot_station_metric_map
    from spatial_vtk.spatial.plot import boxplot
    from spatial_vtk.visualize.waveforms import plot_observed_synthetic_record_section
    register_svtk_cell_timer()


## Configuration

Load the tutorial run scenario, then set the region and corridor choices used below. These choices are examples; you can swap in your own polygon names, event IDs, and corridor dimensions.

In [ ]:
from spatial_vtk.config import notebook_figure_settings, notebook_run_context

config_path = repo_root / "data/examples/configuration/example_spatial_vtk_config.yaml"

# Load the tutorial run scenario and make it active for Spatial-VTK helper calls.
context = notebook_run_context(config_path, run_scenario="tutorial")
cfg = context.cfg

# The tutorial GeoJSON has four example regions: LA Basin, East LA, Santa Monica Mountains, and Glendale.
configured_paths = load_configured_input_paths({"region_geojson": "paths.region_geojson"}, cfg=cfg)
geojson_path = configured_paths["region_geojson"]
waveform_figure_settings = notebook_figure_settings("waveform")
spatial_figure_settings = notebook_figure_settings("spatial")
waveform_sidecars = waveform_figure_settings.sidecars
spatial_sidecars = spatial_figure_settings.sidecars
# Basemaps need contextily plus network access or a local tile cache; enable them with SVTK_ADD_BASEMAP=1.
add_basemap = spatial_figure_settings.add_basemap

value_column = "log2_residual"
passbands = ["1-2 sec", "2-3 sec"]
component = "Z"

# These regions and anchors are chosen from the example GeoJSON and metadata.
boundary_region = "LA Basin"
station_region = "LA Basin"
event_region = "Glendale"
through_anchor_station = "OLI"
outward_event_id = "ci38695658"
corridor_station_region = boundary_region


## Load GeoJSON Regions and Tutorial Tables

Start by loading the prepared tables from the earlier notebooks and the larger metrics snapshot used for plotting examples.

In [ ]:
# Load compact workflow outputs through their configured output groups.
ingest_outputs = output_group("step_01_ingest", cfg=cfg)
step_outputs = output_group("step_05_geojson", cfg=cfg)
ingest_tables = ingest_outputs.load_tables(
    {
        "stations": "prepared_stations_path",
        "events": "prepared_events_path",
        "event_stations": "event_station_path",
    },
    cfg=cfg,
)
plotting_tables = step_outputs.load_tables(
    {"comparison_eligible": "comparison_eligible_path"},
    cfg=cfg,
)
stations = ingest_tables["stations"]
events = ingest_tables["events"]
event_stations = ingest_tables["event_stations"]
comparison_eligible = plotting_tables["comparison_eligible"]

# Load the configured metric snapshot used for figure examples.
configured_inputs = load_configured_input_tables({"metrics": "paths.metric_figure_snapshot"}, cfg=cfg)
metrics = configured_inputs["metrics"]

# Write the configured GeoJSON region summary table through the package workflow helper.
geojson_summary_result = run_geojson_region_summary_workflow_from_config(
    config_path=str(config_path),
    run_scenario="tutorial",
    metrics_table="paths.metric_figure_snapshot",
    geojson_path="paths.region_geojson",
    chunksize=100_000,
    verbose=True,
)
print(geojson_summary_result)

# Use the model label stored in the metric table for plotting filters.
model_name = first_nonempty_table_value(metrics, "model", fallback="model")
model_label = model_display_name(model_name)

# Preview the configured GeoJSON regions through the package helper.
region_preview = geojson_polygon_preview_table(geojson_path)
region_preview


## Visualize the GeoJSON Regions

Plot the polygons with stations and events on the same basemap. This is the quickest way to check that the GeoJSON file overlaps your project area.

In [ ]:
# Plot all configured GeoJSON polygons with station and event context.
geojson_fig = plot_geojson_polygons_map(
    geojson_path,
    stations_df=stations,
    events_df=events,
    add_basemap=add_basemap,
    title="Example GeoJSON Regions",
    showfig=False,
    savefig=True,
    outpath=step_outputs.figure_path(
        "geojson_polygons_map_path",
        stem_parts=("step_05", "geojson_regions"),
    ),
    **spatial_sidecars.kwargs(),
)
display(geojson_fig)
plt.close(geojson_fig)


## Regional PGA Contrast

Add station-region labels from the GeoJSON file, then compare PGA residuals across stations that fall inside one of the example polygons. The table below the plot compares each mapped region against the LA Basin baseline.


In [ ]:
# Add station GeoJSON labels and keep metric rows inside one of the example polygons.
metrics_in_mapped_station_regions = geojson_metric_region_frame(
    metrics,
    geojson_path,
    target="station",
    selector="all",
    region_col="station_region",
)
metrics_by_station_region = metrics_in_mapped_station_regions

# Build a regional contrast boxplot for PGA residuals by station polygon.
pga_region_boxplot = boxplot(
    data=metrics_in_mapped_station_regions,
    dep="PGA",
    indep="station_region",
    value_col=value_column,
    passband=passbands,
    model=model_name,
    component=component,
    compare_to="LA Basin",
    table=True,
    title="PGA Residuals by Station Region",
    showfig=False,
    savefig=True,
    outpath=step_outputs.figure_path(
        "region_boxplot_figure_path",
        stem_parts=("step_05", "pga", "region_boxplot"),
    ),
    **spatial_sidecars.kwargs(),
)
display(pga_region_boxplot)
plt.close(pga_region_boxplot)


## Residual Map for Events and Stations in Different Regions

Here you can ask a more targeted question: for events in one polygon and stations in another, where are the average station residuals high or low? This example uses Glendale events recorded by LA Basin stations.


In [ ]:
# Add event-region labels to the same metric table.
metrics_by_regions = geojson_metric_region_frame(
    metrics_by_station_region,
    geojson_path,
    target="event",
    selector="all",
    require_inside=False,
    region_col="event_region",
)

# Keep PGA rows for Glendale events recorded at LA Basin stations.
regional_pga = geojson_metric_subset_frame(
    metrics_by_regions,
    metric="PGA",
    passband=passbands,
    component=component,
    event_region=event_region,
    station_region=station_region,
)

# Build a PGA residual field for that region-to-region subset.
regional_pga_field = build_metric_field(
    regional_pga,
    "PGA",
    value_column=value_column,
)

# Summarize mean station residuals without removing the event mean.
regional_pga_bias = summarize_station_bias(
    regional_pga_field,
    value_col="field_value",
    center_by_event=False,
    min_events_per_station=1,
)

# Keep the Glendale events visible behind the station residual markers.
regional_event_points = event_rows_for_records(events, regional_pga)

# Map the station-mean PGA residuals with the Glendale and LA Basin polygons as faint context.
regional_pga_map = plot_station_metric_map(
    regional_pga_bias,
    value_col="mean_centered",
    lon_col="lon",
    lat_col="lat",
    events_df=regional_event_points,
    geojson_path=geojson_path,
    add_basemap=add_basemap,
    polygon_selector=[event_region, station_region],
    polygon_alpha=0.12,
    label_polygons=True,
    event_alpha=0.76,
    title="Mean PGA log2(obs/syn) Residual\nEvents in Glendale; Stations in LA Basin",
    showfig=False,
    savefig=True,
    outpath=step_outputs.figure_path(
        "station_metric_map_path",
        stem_parts=("step_05", "pga", "glendale_events", "la_basin_stations"),
    ),
    **spatial_sidecars.kwargs(),
)
display(regional_pga_map)
plt.close(regional_pga_map)


## Define and Visualize Corridors

Corridors describe the part of a polygon boundary or near-boundary area you want to study. The examples below show two common options: a corridor crossing a boundary and a corridor going outward from a boundary.


In [ ]:
# Build a narrow corridor that extends both inside and outside the LA Basin boundary near station OLI.
through_corridors = build_boundary_corridors(
    geojson_path,
    config=BoundaryCorridorConfig(
        selector=boundary_region,
        mode="through_boundary",
        along_boundary_width_km=10,
        inside_length_km=15,
        outside_length_km=20,
        anchor=CorridorAnchorConfig(source="station", strategy="id", id_value=through_anchor_station),
    ),
    station_df=stations,
    event_df=events,
)

# Build a narrow corridor that starts at the LA Basin boundary and goes outward toward event ci38695658.
outward_corridors = build_boundary_corridors(
    geojson_path,
    config=BoundaryCorridorConfig(
        selector=boundary_region,
        mode="outward",
        along_boundary_width_km=10,
        inside_length_km=15,
        outside_length_km=20,
        anchor=CorridorAnchorConfig(source="event", strategy="id", id_value=outward_event_id),
    ),
    station_df=stations,
    event_df=events,
)

# Select paths that pass through the through-boundary corridor.
through_corridor_paths = select_records_by_corridors(
    event_stations,
    through_corridors,
    config=CorridorSelectionConfig(path_filter="passes_through_corridor", min_path_length_km=0.1),
)

# Select paths that pass through the outward corridor.
outward_corridor_paths = select_records_by_corridors(
    event_stations,
    outward_corridors,
    config=CorridorSelectionConfig(path_filter="passes_through_corridor", min_path_length_km=0.1),
)

# Plot the through-boundary corridor with matching event-station paths and the OLI anchor highlighted.
through_corridor_fig = plot_corridor_map(
    through_corridors,
    stations_df=stations,
    events_df=events,
    records_df=corridor_record_pair_frame(through_corridor_paths),
    add_basemap=add_basemap,
    highlight_anchor=True,
    title="Through-Boundary Corridor at the LA Basin Edge\nAnchor: station OLI",
    showfig=False,
    savefig=True,
    outpath=step_outputs.figure_path(
        "corridor_map_path",
        stem_parts=("step_05", "corridor", "through_boundary"),
    ),
    **spatial_sidecars.kwargs(),
)
display(through_corridor_fig)
plt.close(through_corridor_fig)

# Plot the outward corridor with the paths that pass through its footprint and the event anchor highlighted.
outward_corridor_fig = plot_corridor_map(
    outward_corridors,
    stations_df=stations,
    events_df=events,
    records_df=corridor_record_pair_frame(outward_corridor_paths),
    add_basemap=add_basemap,
    highlight_anchor=True,
    title="Outward Corridor from the LA Basin Boundary\nAnchor: event ci38695658",
    showfig=False,
    savefig=True,
    outpath=step_outputs.figure_path(
        "corridor_map_path",
        stem_parts=("step_05", "corridor", "outward"),
    ),
    **spatial_sidecars.kwargs(),
)
display(outward_corridor_fig)
plt.close(outward_corridor_fig)


## Record Sections for Boundary-Crossing Paths

Now combine the polygon crossing test with the corridor selection. This cell keeps paths that cross the LA Basin boundary within the through-boundary corridor, then loads the QC-passed observed and synthetic waveforms for those paths.

In [ ]:
# Classify paths by whether they cross the LA Basin polygon boundary.
central_boundary_paths = classify_paths_with_geojson(
    event_stations,
    geojson_path,
    relation="crosses_boundary",
    selector=boundary_region,
    direction="either",
)

# Keep only paths that cross the boundary and pass through the through-boundary corridor.
boundary_crossing_paths = select_records_by_corridors(
    geojson_matched_record_frame(central_boundary_paths),
    through_corridors,
    config=CorridorSelectionConfig(path_filter="passes_through_corridor", min_path_length_km=0.1),
)
selected_eligible = event_station_records_matching_pairs(comparison_eligible, boundary_crossing_paths)

# Load QC-passed R-component observed/synthetic waveform pairs for the selected boundary-crossing paths.
boundary_waveforms = build_qc_waveform_comparison_records(
    event_stations,
    comparison_eligible=selected_eligible,
    component="R",
    passband="1-2 sec",
    max_distance_km=None,
    max_records=12,
)

# Plot the observed and synthetic records for the selected boundary-crossing paths.
boundary_record_section = plot_observed_synthetic_record_section(
    boundary_waveforms,
    components=["R"],
    normalize=True,
    scale=2.5,
    title="Observed vs Synthetic Records for LA Basin Boundary-Crossing Paths",
    filter_label="lowpass 1 Hz; R component; 1-2 sec QC passband",
    time_limit_s=60,
    showfig=False,
    savefig=True,
    outpath=step_outputs.figure_path(
        "record_section_figure_path",
        stem_parts=("step_05", "boundary_crossing_record_section"),
    ),
    **waveform_sidecars.kwargs(),
)
display(boundary_record_section)
plt.close(boundary_record_section)

# Show the first few selected paths used by the record section.
corridor_record_preview_frame(boundary_crossing_paths)


## PGV Residual Map for an Outward Corridor

Finally, select events inside the outward corridor and stations inside the LA Basin polygon. The map shows the station-mean PGV residuals for that corridor-based subset.


In [ ]:
# Select event-station rows whose events fall inside the outward corridor.
outward_event_paths = select_records_by_corridors(
    event_stations,
    outward_corridors,
    config=CorridorSelectionConfig(event_filter="inside_corridor"),
)
outward_event_ids = event_ids_from_records(outward_event_paths)
events_in_outward_corridor = event_rows_for_records(events, event_ids=outward_event_ids)
selected_event_names = event_label_preview_frame(events_in_outward_corridor)

display(Markdown("### Events inside the outward corridor"))
display(selected_event_names)

# Keep PGV rows for events inside the outward corridor and stations inside the LA Basin polygon.
pgv_corridor_rows = geojson_metric_subset_frame(
    metrics_by_regions,
    metric="PGV",
    passband=passbands,
    component=component,
    event_ids=outward_event_ids,
    station_region=corridor_station_region,
)

# Build a PGV residual field for the corridor-selected subset.
pgv_corridor_field = build_metric_field(
    pgv_corridor_rows,
    "PGV",
    value_column=value_column,
)

# Summarize mean station residuals without removing the event mean.
pgv_corridor_bias = summarize_station_bias(
    pgv_corridor_field,
    value_col="field_value",
    center_by_event=False,
    min_events_per_station=1,
)

# Keep the plotted event-station paths for the same events and LA Basin stations shown on the residual map.
pgv_corridor_paths = event_station_records_matching_pairs(outward_event_paths, pgv_corridor_rows)

# Map station-mean PGV residuals with the outward corridor, selected events, and selected paths overlaid.
pgv_corridor_map = plot_station_metric_map(
    pgv_corridor_bias,
    value_col="mean_centered",
    lon_col="lon",
    lat_col="lat",
    geojson_path=geojson_path,
    polygon_selector="all",
    polygon_alpha=0.10,
    label_polygons=True,
    add_basemap=add_basemap,
    corridors_df=outward_corridors,
    events_df=events_in_outward_corridor,
    records_df=corridor_record_pair_frame(pgv_corridor_paths),
    title="Mean PGV log2(obs/syn) Residual\nEvents in Outward Corridor; Stations in LA Basin",
    showfig=False,
    savefig=True,
    outpath=step_outputs.figure_path(
        "station_metric_map_path",
        stem_parts=("step_05", "pgv", "outward_corridor", "la_basin_stations"),
    ),
    **spatial_sidecars.kwargs(),
)
display(pgv_corridor_map)
plt.close(pgv_corridor_map)
